In [1]:
import sys
import os
from datetime import datetime, timedelta, timezone

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import pandas as pd
import polars as pl
import MetaTrader5 as mt5
from src.infra.mtBase import mtBase
from src.infra.PredictionParser import PredictionParser, PredictionData

In [2]:
mtb = mtBase(
    account="darwinexzero_acc",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)

In [3]:
mtb.mt5_init()

MetaTrader 5 connection established


In [12]:
symbol = "MU"
seconds_back = 1200  # last 10 seconds

In [6]:
syminfo = mtb.get_symbol_info(symbol)
curtime = pd.to_datetime(syminfo["time"], unit="s", utc=True)

from_time = curtime - pd.Timedelta(seconds=seconds_back)

# Get ticks in that range
ticks = mt5.copy_ticks_range(symbol, from_time, curtime, mt5.COPY_TICKS_ALL)

df = pd.DataFrame(ticks)

# Convert timestamps to readable datetimes (time is seconds; time_msc is milliseconds)
if not df.empty:
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["time_msc"] = pd.to_datetime(df["time_msc"], unit="ms", utc=True)

df.tail()

,time,bid,ask,last,volume,time_msc,flags,volume_real
164,2026-01-27 22:58:54+00:00,409.91,410.19,0.0,0,2026-01-27 22:58:54.579000+00:00,1154,0.0
165,2026-01-27 22:58:54+00:00,409.87,410.19,0.0,0,2026-01-27 22:58:54.782000+00:00,1154,0.0
166,2026-01-27 22:58:54+00:00,409.81,410.13,0.0,0,2026-01-27 22:58:54.980000+00:00,1158,0.0
167,2026-01-27 22:58:55+00:00,409.81,410.19,0.0,0,2026-01-27 22:58:55.080000+00:00,1028,0.0
168,2026-01-27 22:58:55+00:00,409.88,410.19,0.0,0,2026-01-27 22:58:55.980000+00:00,1154,0.0


In [14]:
syminfo = mtb.get_symbol_info(symbol)
curtime = pd.to_datetime(syminfo["time"], unit="s", utc=True)

from_time = curtime - pd.Timedelta(seconds=seconds_back)

# Get ticks in that range
ticks = mt5.copy_ticks_range(symbol, from_time, curtime, mt5.COPY_TICKS_ALL)

df = pd.DataFrame(ticks)

# Convert timestamps to readable datetimes (time is seconds; time_msc is milliseconds)
if not df.empty:
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["time_msc"] = pd.to_datetime(df["time_msc"], unit="ms", utc=True)

df

,time,bid,ask,last,volume,time_msc,flags,volume_real
0,2026-01-28 16:32:00+00:00,421.37,422.12,0.0,0,2026-01-28 16:32:00.152000+00:00,1030,0.0
1,2026-01-28 16:32:00+00:00,421.37,422.61,0.0,0,2026-01-28 16:32:00.249000+00:00,1028,0.0
2,2026-01-28 16:32:00+00:00,421.37,422.98,0.0,0,2026-01-28 16:32:00.450000+00:00,1028,0.0
3,2026-01-28 16:32:00+00:00,421.37,422.03,0.0,0,2026-01-28 16:32:00.853000+00:00,1028,0.0
4,2026-01-28 16:32:01+00:00,421.37,422.39,0.0,0,2026-01-28 16:32:01.149000+00:00,1028,0.0
...,...,...,...,...,...,...,...,...
1278,2026-01-28 16:39:19+00:00,429.33,429.72,0.0,0,2026-01-28 16:39:19.325000+00:00,1030,0.0
1279,2026-01-28 16:39:19+00:00,429.22,429.72,0.0,0,2026-01-28 16:39:19.924000+00:00,1026,0.0
1280,2026-01-28 16:39:20+00:00,429.08,429.72,0.0,0,2026-01-28 16:39:20.324000+00:00,1026,0.0
1281,2026-01-28 16:39:20+00:00,429.00,429.72,0.0,0,2026-01-28 16:39:20.824000+00:00,1026,0.0


In [4]:
ac_info = mtb.get_account_info()._asdict()
print(ac_info)

{'login': 5041376226, 'trade_mode': 0, 'leverage': 100, 'limit_orders': 100, 'margin_so_mode': 0, 'trade_allowed': True, 'trade_expert': True, 'margin_mode': 0, 'currency_digits': 2, 'fifo_close': False, 'balance': 1000.0, 'credit': 0.0, 'profit': 45.4, 'equity': 1045.4, 'margin': 372.7, 'margin_free': 672.7, 'margin_level': 280.49369466058494, 'margin_so_call': 50.0, 'margin_so_so': 30.0, 'margin_initial': 0.0, 'margin_maintenance': 0.0, 'assets': 0.0, 'liabilities': 0.0, 'commission_blocked': 0.0, 'name': 'Kthim Imeri', 'server': 'MetaQuotes-Demo', 'currency': 'USD', 'company': 'MetaQuotes Ltd.'}


In [3]:
last_error = mt5.last_error()
#mtb.shutdown()
a = (mt5.account_info().login)
print(a)
print(type(a))
term_info = mt5.terminal_info()._asdict()

5041376226
<class 'int'>


In [15]:
# Get and print all positions
pos_df: pl.DataFrame = mtb.get_position_df()
if pos_df.is_empty():
    print("No open positions.")
else:
    print(pos_df.to_pandas())

ord_df = mtb.get_orders_df()
if ord_df.is_empty():
    print("No open orders.")
else:
    print(ord_df.to_pandas())


        ticket        time       time_msc  time_update  time_update_msc  type  \
0  53657798221  1760633301  1760633301123   1760633301    1760633301123     0   

   magic   identifier  reason  volume  price_open    sl   tp  price_current  \
0      0  53657798221       0    10.0       37.27  32.0  0.0          37.01   

   swap  profit symbol comment external_id                 time_dt  
0   0.0    -2.6   INTC                     2025-10-16 16:48:21.123  
No open orders.


In [ ]:
for i in range(10):
    acc = mtb.get_account_info()
    if acc is None:
        print("Failed to get account info")
    else:
        pass #print(f"Balance: {acc.balance}, Equity: {acc.equity}, Free Margin: {acc.margin_free}")

In [8]:
for i in range(200):
    pos_df = mtb.get_position_df()
    if pos_df is None or pos_df.is_empty():
        print("No open positions.")
    else:
        pass

In [6]:
res = mtb.get_symbol_tick("MSFT")
print(res)

None


In [9]:
res = mtb.get_symbol_tick("USDCAD")
print(res)

None
